# 12 MLA_From_Intuition_To_Implementation

## Problem

当 GQA 仍然不够省时，MLA 想解决什么问题？它和普通 K/V cache 思路最关键的区别在哪里？

## Dependencies

建议先理解 KV cache、MQA、GQA 的瓶颈来源。


## Module_Goals

- 理解 MLA 的核心动机：进一步降低注意力状态缓存成本
- 理解 latent compression 的核心直觉
- 理解“先压缩存储，再映射回需要的表示空间”的思路
- 会写一个最小 MLA forward 过程

## Scope_And_Approach

这个 notebook 会先用直觉和小例子说明问题，再给出最小实现。重点是把模块为什么存在、输入输出是什么、关键步骤如何衔接说明清楚。


## 先把问题说透

GQA 已经减少了 K/V 头数，但只要上下文很长、层数很多、batch 很大，KV cache 仍然会吃掉大量显存和带宽。于是问题变成：

**我们能不能不直接缓存那么多原始 K/V 表示，而是缓存一个更紧凑的 latent 状态，需要时再映射回来？**

这就是理解 MLA 最重要的一步。

可以把它类比成压缩文件：

- 不是把每一页内容都原样存下来。
- 先提取一个更紧凑的表示。
- 需要使用时，再恢复到适合计算的空间。

这里要特别注意：我们关注的是思路，不是严格复刻某个工业实现的每一条细节。


In [ ]:
import numpy as np

np.random.seed(123)
np.set_printoptions(precision=3, suppress=True)

seq_len = 4
hidden_dim = 8
latent_dim = 3
head_dim = 4
num_heads = 2

x = np.random.randn(seq_len, hidden_dim) * 0.2

W_latent = np.random.randn(hidden_dim, latent_dim) * 0.3
W_k_up = np.random.randn(latent_dim, num_heads * head_dim) * 0.3
W_v_up = np.random.randn(latent_dim, num_heads * head_dim) * 0.3
W_q = np.random.randn(hidden_dim, num_heads * head_dim) * 0.3


In [ ]:
# 第一步：把每个 token 压到更小的 latent 空间
latent_cache = x @ W_latent
print('latent_cache.shape =', latent_cache.shape)  # (seq_len, latent_dim)

# 第二步：需要做注意力时，再把 latent 映射回 K/V 所需空间
K = (latent_cache @ W_k_up).reshape(seq_len, num_heads, head_dim)
V = (latent_cache @ W_v_up).reshape(seq_len, num_heads, head_dim)
Q = (x @ W_q).reshape(seq_len, num_heads, head_dim)

print('K.shape =', K.shape)
print('V.shape =', V.shape)
print('Q.shape =', Q.shape)

raw_kv_cache_size = seq_len * num_heads * head_dim * 2
latent_cache_size = seq_len * latent_dim
print('raw_kv_cache_size    =', raw_kv_cache_size)
print('latent_cache_size    =', latent_cache_size)


In [ ]:
def softmax(logits):
    shifted = logits - np.max(logits, axis=-1, keepdims=True)
    exp = np.exp(shifted)
    return exp / np.sum(exp, axis=-1, keepdims=True)

outputs = []
for h in range(num_heads):
    q = Q[:, h, :]
    k = K[:, h, :]
    v = V[:, h, :]
    scores = (q @ k.T) / np.sqrt(head_dim)
    mask = np.triu(np.ones_like(scores), k=1) * -1e9
    weights = softmax(scores + mask)
    outputs.append(weights @ v)

mla_output = np.concatenate(outputs, axis=-1)
print('mla_output.shape =', mla_output.shape)
print(mla_output)


## Trade_Offs_And_Risk_Points

- 把 MLA 误解成“又一种换名字的 GQA”。它更关键的点在于缓存表示的压缩与重构思路。
- 试图一开始就对齐工业实现细节，结果反而看不清核心思想。
- 只关注省显存，不去想重新映射带来的额外计算和设计复杂度。

## Checkpoints

- 把 `latent_dim` 改成 2、4、6，观察 cache 大小与表达空间之间的变化。
- 思考为什么“压缩得越狠越好”并不成立。
- 尝试用自己的话解释 MLA 和 GQA 的主要差别。

## Summary

MLA 的分析价值很高，因为它把大模型工程里的一个核心现实摆到了台面上：推理瓶颈不只来自算力，也来自状态存储和带宽。理解这一点，才会更容易看清 DeepSeek 一类设计为什么重要。

## Next_Module

下一节进入 MoE。那是另一条非常重要的现代大模型扩容路线：不是让每个 token 都经过所有参数，而是有选择地激活专家。
